In [2]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
import polars as pl
import os
import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
print(os.getcwd())

c:\Users\eliot.nehme\repositories\memoire-ia\codes


In [4]:
# avant de décider du type de chaque colonne. 
# C'est un peu plus lent (quelques secondes), mais c'est 100% sûr.

table_sinistres = pl.read_csv(
    "../data/car_insurance_fraud_dataset.csv", 
    infer_schema_length=None  # <--- Le secret est là !
)

print(f"Dimensions : {table_sinistres.shape}") # Affiche le nombre de lignes/colonnes
print(table_sinistres.head())

Dimensions : (30000, 24)
shape: (5, 24)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ policy_id ┆ policy_st ┆ policy_de ┆ policy_an ┆ … ┆ police_re ┆ claim_amo ┆ total_cla ┆ fraud_re │
│ ---       ┆ ate       ┆ ductible  ┆ nual_prem ┆   ┆ port_avai ┆ unt       ┆ im_amount ┆ ported   │
│ str       ┆ ---       ┆ ---       ┆ ium       ┆   ┆ lable     ┆ ---       ┆ ---       ┆ ---      │
│           ┆ str       ┆ i64       ┆ ---       ┆   ┆ ---       ┆ f64       ┆ f64       ┆ str      │
│           ┆           ┆           ┆ f64       ┆   ┆ str       ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ POL100000 ┆ GA        ┆ 400       ┆ 1430.78   ┆ … ┆ Yes       ┆ 8161.36   ┆ 11677.6   ┆ Y        │
│ POL100001 ┆ PA        ┆ 300       ┆ 854.49    ┆ … ┆ No        ┆ 18561.79  ┆ 18027.81  ┆ N        │
│ POL100002 ┆ MI        ┆ 400       ┆ 1247.28   ┆ …

In [14]:
# 1. Charger les données
df = pd.read_csv("../data/car_insurance_fraud_dataset.csv")

# 2. Nettoyage de base
# La colonne 'authorities_contacted' avait des valeurs manquantes, on les remplace par 'None'
df['authorities_contacted'] = df['authorities_contacted'].fillna('None')

# On supprime 'policy_id' et 'incident_date' car l'ID n'a pas de pouvoir prédictif 
# (Pour aller plus loin, on pourrait extraire le mois/jour de la date, mais on simplifie pour l'instant)
df = df.drop(columns=['policy_id', 'incident_date'])

# 3. Transformer la variable cible en 0 et 1 (N=0, Y=1)
df['fraud_reported'] = df['fraud_reported'].map({'N': 0, 'Y': 1})

# 4. Encodage des variables catégorielles (texte -> chiffres)
categorical_cols = df.select_dtypes(include=['object']).columns
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# 5. Séparer les variables explicatives (X) et la cible (y)
X = df_encoded.drop('fraud_reported', axis=1)
y = df_encoded['fraud_reported']

# 6. Séparer en Train (80%) et Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 7. Normaliser les données numériques (mettre toutes les valeurs à la même échelle)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- NOUVELLE ÉTAPE 7 bis : SMOTE ---
# On rééquilibre les classes en créant des données synthétiques de fraude
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# --- ÉTAPE 8 MODIFIÉE ---
# Entraînement du modèle (Random Forest) sur les données RÉÉQUILIBRÉES
# (On peut enlever le class_weight='balanced' car SMOTE a déjà fait le travail)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_resampled, y_train_resampled) # <-- Attention à bien utiliser les variables _resampled ici !

# 9. Prédictions sur le jeu de test
y_pred = rf_model.predict(X_test_scaled)

# 10. Évaluation des résultats
print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred))
print("\nRapport de classification :")
print(classification_report(y_test, y_pred))

C:\Users\eliot.nehme\AppData\Local\Temp\ipykernel_14476\3158139207.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns


Matrice de confusion :
[[5312    0]
 [ 688    0]]

Rapport de classification :
              precision    recall  f1-score   support

           0       0.89      1.00      0.94      5312
           1       0.00      0.00      0.00       688

    accuracy                           0.89      6000
   macro avg       0.44      0.50      0.47      6000
weighted avg       0.78      0.89      0.83      6000



c:\Users\eliot.nehme\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\eliot.nehme\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\eliot.nehme\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio